# Library & Data

In [ ]:
# If running in Colab, mount Google Drive here.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
train=pd.read_csv('../data/train.csv')
# test=pd.read_csv('../data/test.csv')

# 초기) 간단 EDA

In [ ]:
print("Train의 형태: ",train.shape)

In [ ]:
train.info()

In [ ]:
train.head()

In [ ]:
pd.DataFrame(train['full_log'].value_counts())

In [ ]:
#train full_log의 길이 확인
train['full_log'].str.split(' ').str.len().hist(bins=50)

In [ ]:
#train level별 값 확인
train['level'].value_counts()

# 기본) 레벨에 따른 단어 분리

In [ ]:
train

In [ ]:
class letssee:
  def __init__(self, data):
    train_df = data

    first_strings = pd.merge(train_df.loc[train_df.level!=0].full_log.apply(lambda x: x.split(' ')[0]).value_counts().rename('lv!=0'),
                         train_df.loc[train_df.level==0].full_log.apply(lambda x: x.split(' ')[0]).value_counts().rename('lv==0'),
                         how='outer',left_index=True,right_index=True).fillna(0).astype(int)

    self.first_strings = first_strings.sort_values('lv==0',ascending=False)

  def graph(self): 
    first_strings = self.first_strings
    fig,ax = plt.subplots(1,1,figsize=(12,7))
    ax.bar([i-0.25 for i in range(first_strings.shape[0])], first_strings.values[:,1],width=0.4)
    ax.bar([i+0.25 for i in range(first_strings.shape[0])], first_strings.values[:,0],width=0.4)
    ax.set_yscale("log")
    ax.set_xticks(range(first_strings.shape[0]))
    ax.set_xticklabels(first_strings.index, rotation=70)
    ax.legend(['Level == 0','Level != 0'])
    fig.show()

  def show(self):
    first_strings = self.first_strings
    print(first_strings)


In [ ]:
a = letssee(train)

In [ ]:
a.graph()

In [ ]:
a.show()

# 심화1) 숫자가 의미가 있을까?
* https://dacon.io/competitions/official/235717/codeshare/2637?page=1&dtype=recent 참조

In [ ]:
# 기존
pd.DataFrame(train['full_log'].value_counts())

In [ ]:
train1 = train.copy()
# text 전처리로 문자만 남김

train1['full_log'] = train1['full_log'].str.replace(pat=r'[^A-Za-z]', repl=r'', regex=True)

In [ ]:
# 문자 제거
train1['full_log'].value_counts()

동일한 Text가 다수 존재 (클러스터링)

# 심화1-1) 동일한 Text는 어떤 의미를 가지는 가?

In [ ]:
train_df = train1
b = letssee(train_df)

In [ ]:
b.show()

* 동일한 Text는 동일한 Level을 가짐
* 숫자는 크게 중요하지 않음

# 심화2) 숫자와 월을 변경해보자

In [ ]:
train2 = train.copy()

#full_log에서 숫자는 마스킹 처리
train2['full_log']=train2['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', 'na')

In [ ]:
# 숫자와 해당 월을 제거 후
train2['full_log'].value_counts()

In [ ]:
c = letssee(train2)

In [ ]:
c.graph()

In [ ]:
c.show()

# 심화2-1) 숫자와 월을 제거해보자

In [ ]:
train2 = train.copy()

#full_log에서 숫자는 마스킹 처리
train2['full_log']=train2['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

In [ ]:
# 숫자와 해당 월을 제거 후
train2['full_log'].value_counts()

In [ ]:
c = letssee(train2)

In [ ]:
c.graph()

In [ ]:
c.show()

# 심화2-2) 여러가지 조합

In [ ]:
train2 = train.copy()

#full_log에서 숫자는 마스킹 처리
train2['full_log']=train2['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

In [ ]:
# 문자열 제거
train2['full_log'] = train2['full_log'].str.replace(pat=r'[^A-Za-z]', repl=r' ', regex=True)

In [ ]:
# 숫자와 해당 월을 제거 후
train2['full_log'].value_counts()

In [ ]:
c = letssee(train2)

In [ ]:
c.graph()

In [ ]:
c.show()

> 대회 규칙이 "패턴 매칭 알고리즘 사용 불가"
* 패턴 매칭 알고리즘 사용 불가는 규칙기반 모델이 아닌 학습기반의 모델만 사용가능하다는 의미
* tag를 추출하여 학습 feature로 사용하는 것은 가능하나 추출한 tag와 사람이 지정한 조건문만을 이용한 예측은 불가